## 4. NMF

### 4.1 NMF Grid search

In [ ]:
# ==== Préparation ====
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)

# ==== Coherence ====
def compute_coherence(topics):
    cm = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

# ==== Diversity ====
def compute_topic_diversity(topics, topk=10):
    all_words = [word for topic in topics for word in topic[:topk]]
    return len(set(all_words)) / len(all_words) if all_words else 0

# ==== Extraction topics ====
def get_topics(model, feature_names, topk=10):
    topics = []
    for topic_weights in model.components_:
        idx = topic_weights.argsort()[-topk:][::-1]
        topics.append([feature_names[i] for i in idx])
    return topics

# ==== Grilles ====
n_topics_list = range(8,20)
ngram_range_list = [(1,1), (1,2)]

results = []
models = {}

# ==== Grid search ====
for n_topics, ngram_range in product(n_topics_list, ngram_range_list):

    # TF-IDF
    vectorizer = TfidfVectorizer(
        stop_words=STOPWORDS,
        ngram_range=ngram_range,
        min_df=0.005,
        max_df=0.7
    )

    X = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()

    # NMF
    nmf_model = NMF(
        n_components=n_topics,
        random_state=42,
        init="nndsvd"
    )

    W = nmf_model.fit_transform(X)

    # Topics
    topics = get_topics(nmf_model, feature_names)

    # Metrics
    coh = compute_coherence(topics)
    div = compute_topic_diversity(topics)
    score = coh * div

    # Stockage
    key = (n_topics, ngram_range)
    models[key] = (nmf_model, vectorizer, W)

    results.append({
        "n_topics": n_topics,
        "ngram_range": ngram_range,
        "coherence": coh,
        "diversity": div,
        "score": score
    })

    print(f"topics={n_topics}, ngram={ngram_range} | "
          f"coh={coh:.4f}, div={div:.4f}, score={score:.4f}")

# ==== Résultats ====
df_grid = pd.DataFrame(results)
df_grid = df_grid.sort_values("score", ascending=False).reset_index(drop=True)

print("\nTop modèles :")
print(df_grid)

df_grid.to_csv("nmf_grid_search_simple.csv", sep=";", index=False)

topics=8, ngram=(1, 1) | coh=0.4834, div=0.9000, score=0.4351
topics=8, ngram=(1, 2) | coh=0.4995, div=0.8750, score=0.4371
topics=9, ngram=(1, 1) | coh=0.4838, div=0.8889, score=0.4301
topics=9, ngram=(1, 2) | coh=0.5058, div=0.8667, score=0.4384
topics=10, ngram=(1, 1) | coh=0.4864, div=0.8300, score=0.4037
topics=10, ngram=(1, 2) | coh=0.5191, div=0.8700, score=0.4516
topics=11, ngram=(1, 1) | coh=0.4968, div=0.8182, score=0.4065
topics=11, ngram=(1, 2) | coh=0.5309, div=0.8636, score=0.4585
topics=12, ngram=(1, 1) | coh=0.5017, div=0.8250, score=0.4139
topics=12, ngram=(1, 2) | coh=0.5501, div=0.8750, score=0.4813
topics=13, ngram=(1, 1) | coh=0.5102, div=0.8462, score=0.4317
topics=13, ngram=(1, 2) | coh=0.5601, div=0.8615, score=0.4826
topics=14, ngram=(1, 1) | coh=0.5132, div=0.8357, score=0.4289
topics=14, ngram=(1, 2) | coh=0.5566, div=0.8786, score=0.4890
topics=15, ngram=(1, 1) | coh=0.5139, div=0.8133, score=0.4180
topics=15, ngram=(1, 2) | coh=0.5787, div=0.8800, score=0.5